In [2]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))
from src.data.filter import load_data, filter_with_nation_winners

In [7]:
def is_extra(delivery):
        if 'extras' not in delivery:
            return False
        return 'noballs' in delivery['extras'] or 'wides' in delivery['extras']

def process_first_innings(innings):
    states = []
    runs = 0
    wickets = 0
    balls = 0
    
    for over in innings['overs']:
        for delivery in over['deliveries']:
            runs += delivery['runs']['total']
            wickets += 1 if 'wicket' in delivery else 0
            balls += 0 if is_extra(delivery) else 1
            states.append({ 'runs': runs, 'wickets': wickets, 'balls': balls })
    return states

def process_second_innings(innings):
    assert 'target' in innings, "Second innings data must contain a target"
    states = []
    runs = innings['target']['runs']
    wickets = 10
    balls = 300

    for over in innings['overs']:
        for delivery in over['deliveries']:
            runs -= delivery['runs']['total']
            wickets -= 1 if 'wicket' in delivery else 0
            balls -= 0 if is_extra(delivery) else 1
            states.append({ 'runs': runs, 'wickets': wickets, 'balls': balls })
    return states

def insert_labels(inning_states, label):
    for state in inning_states:
        state['label'] = label
    return inning_states

def process_data(data):
    first_innings_data, second_innings_data = [], []
    for d in data:
        first_innings_states = process_first_innings(d['innings'][0])
        second_innings_states = process_second_innings(d['innings'][1])

        first_innings_team = d['innings'][0]['team']
        winning_team = d['info']['outcome']['winner']
        label = 1 if first_innings_team == winning_team else 0

        first_innings_states = insert_labels(first_innings_states, label)
        second_innings_states = insert_labels(second_innings_states, label)

        first_innings_data.extend(first_innings_states)
        second_innings_data.extend(second_innings_states)

    return first_innings_data, second_innings_data


In [ ]:
data = load_data()
data = filter_with_nation_winners(data)
processed_data = process_data(data)

In [12]:
len(processed_data[1])

524790